# 03 — Feature Engineering

## Mục tiêu
- Trích xuất tên quận từ địa chỉ
- Parse biểu đồ giá khu vực thành các features số
- Trích xuất features nhị phân từ tiêu đề + mô tả bằng Regex
- Tạo biến log cho giá bán
- Lưu dataset sẵn sàng cho modeling


In [1]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_parquet("../data/processed/dataset_cleaned.parquet")

## 1. Trích xuất tên Quận từ địa chỉ


In [3]:
import re

def extract_quan(value):
    if pd.isnull(value):
        return value

    if isinstance(value, str):
        value = value.lower()
        
        match = re.search(r"quận\s([^,]+)", value)
        if match:
            value = match.group(1)

        value = value.strip().title()
        
        if value == "Gò Vấp Thanh Phố Ho Chi Minh":
            value = "Gò Vấp"
        
        return value

    return value


df["quan"] = df["dia_chi"].apply(extract_quan)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7961 entries, 0 to 7960
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   giay_to_phap_ly  7961 non-null   object 
 1   dien_tich        7961 non-null   float64
 2   gia_ban          7961 non-null   float64
 3   loai_hinh        7961 non-null   object 
 4   so_phong_ngu     7961 non-null   int64  
 5   mo_ta            7961 non-null   object 
 6   tieu_de          7961 non-null   object 
 7   dia_chi          7961 non-null   object 
 8   bieu_do_gia      7961 non-null   object 
 9   quan             7961 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 622.1+ KB


In [5]:
df["quan"].value_counts()

quan
Gò Vấp        4179
Bình Thạnh    2574
Phú Nhuận     1208
Name: count, dtype: int64

### Nhận xét
- **Gò Vấp** chiếm đa số (4,179 tin, ~52.5%)
- **Bình Thạnh** có 2,574 tin (~32.3%)
- **Phú Nhuận** ít nhất (1,208 tin, ~15.2%)

Sự mất cân bằng này cần lưu ý khi đánh giá model theo quận.


In [6]:
df.head()

,giay_to_phap_ly,dien_tich,gia_ban,loai_hinh,so_phong_ngu,mo_ta,tieu_de,dia_chi,bieu_do_gia,quan
0,đã có sổ,36.0,3.85,"Nhà ngõ, hẻm",2,🍀VÂN KIỀU AN CƯ🍀\r\n\r\n🏡 Nhà Lê Quang Định P5...,🍀VÂN KIỀU AN CƯ🍀 40m2 - Hẻm Ô Tô - Gần mặt tiề...,"Đường Lê Quang Định, Phường 7, Quận Bình Thạnh...","[134.04, 125.8, 125.8, 125.8, 138.64, 138.64, ...",Bình Thạnh
1,chưa xác định,62.0,9.79,"Nhà ngõ, hẻm",4,"CHỈ 9.XTY - NHÀ MỚI KENG,XE HƠI NGỦ NHÀ GIÁP B...","9.xty,Nhà Hẻm Xe Hơi,Xây Mới Khu VIP giáp Phạm...","Đường Phạm Văn Đồng, Phường 13, Quận Bình Thạn...","[133.33, 135.54, 130.95, 130.02, 130.02, 136.3...",Bình Thạnh
2,đã có sổ,54.0,7.20,"Nhà ngõ, hẻm",3,XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,"Đường Nơ Trang Long, Phường 13, Quận Bình Thạn...","[168.37, 168.37, 120.71, 120.71, 124.24, 124.5...",Bình Thạnh
3,đã có sổ,83.0,8.50,"Nhà ngõ, hẻm",3,#e45\r\n🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TI...,🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TIỆN XÂY M...,"Đường Xô Viết Nghệ Tĩnh, Phường 21, Quận Bình ...","[159.22, 159.22, 157.45, 157.45, 135.08, 122.2...",Bình Thạnh
4,đã có sổ,18.0,2.85,"Nhà ngõ, hẻm",2,Nhà hẻm cách đường mặt tiền 20m\r\nNhà ở khu v...,Nhà hẻm cách đường mặt tiền 20m,"Đường Chu Văn An, Phường 12, Quận Bình Thạnh, ...","[132.67, 132.67, 169.64, 169.64, 133.33, 133.3...",Bình Thạnh


## 2. Parse biểu đồ giá khu vực


In [7]:
import ast

def parse_bieu_do_gia(val):
    try:
        lst = ast.literal_eval(val)
        if isinstance(lst, list) and len(lst) >= 2:
            return lst
    except:
        return []

df["bdg_parsed"] = df["bieu_do_gia"].apply(parse_bieu_do_gia)
valid = df["bdg_parsed"].notna()  # add data tạm

df.loc[valid, "gia_kv_hien_tai"] = df.loc[valid, "bdg_parsed"].apply(lambda x: x[-1])
df.loc[valid, "gia_kv_mean"]     = df.loc[valid, "bdg_parsed"].apply(lambda x: np.mean(x))
df.loc[valid, "gia_kv_trend"]    = df.loc[valid, "bdg_parsed"].apply(lambda x: x[-1] - x[0])
df.loc[valid, "gia_kv_volatility"] = df.loc[valid, "bdg_parsed"].apply(lambda x: np.std(x))

df = df.drop(columns=["bdg_parsed"]) # xóa data tạm

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7961 entries, 0 to 7960
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   giay_to_phap_ly    7961 non-null   object 
 1   dien_tich          7961 non-null   float64
 2   gia_ban            7961 non-null   float64
 3   loai_hinh          7961 non-null   object 
 4   so_phong_ngu       7961 non-null   int64  
 5   mo_ta              7961 non-null   object 
 6   tieu_de            7961 non-null   object 
 7   dia_chi            7961 non-null   object 
 8   bieu_do_gia        7961 non-null   object 
 9   quan               7961 non-null   object 
 10  gia_kv_hien_tai    6979 non-null   float64
 11  gia_kv_mean        6979 non-null   float64
 12  gia_kv_trend       6979 non-null   float64
 13  gia_kv_volatility  6979 non-null   float64
dtypes: float64(6), int64(1), object(7)
memory usage: 870.9+ KB


In [9]:
df[~df[["gia_kv_hien_tai", "gia_kv_mean", "gia_kv_trend", "gia_kv_volatility"]].isnull().any(axis=1)].head()

,giay_to_phap_ly,dien_tich,gia_ban,loai_hinh,so_phong_ngu,mo_ta,tieu_de,dia_chi,bieu_do_gia,quan,gia_kv_hien_tai,gia_kv_mean,gia_kv_trend,gia_kv_volatility
0,đã có sổ,36.0,3.85,"Nhà ngõ, hẻm",2,🍀VÂN KIỀU AN CƯ🍀\r\n\r\n🏡 Nhà Lê Quang Định P5...,🍀VÂN KIỀU AN CƯ🍀 40m2 - Hẻm Ô Tô - Gần mặt tiề...,"Đường Lê Quang Định, Phường 7, Quận Bình Thạnh...","[134.04, 125.8, 125.8, 125.8, 138.64, 138.64, ...",Bình Thạnh,140.91,142.331538,6.87,13.421004
1,chưa xác định,62.0,9.79,"Nhà ngõ, hẻm",4,"CHỈ 9.XTY - NHÀ MỚI KENG,XE HƠI NGỦ NHÀ GIÁP B...","9.xty,Nhà Hẻm Xe Hơi,Xây Mới Khu VIP giáp Phạm...","Đường Phạm Văn Đồng, Phường 13, Quận Bình Thạn...","[133.33, 135.54, 130.95, 130.02, 130.02, 136.3...",Bình Thạnh,140.00,137.227692,6.67,4.625099
2,đã có sổ,54.0,7.20,"Nhà ngõ, hẻm",3,XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,"Đường Nơ Trang Long, Phường 13, Quận Bình Thạn...","[168.37, 168.37, 120.71, 120.71, 124.24, 124.5...",Bình Thạnh,151.05,136.483846,-17.32,18.098514
3,đã có sổ,83.0,8.50,"Nhà ngõ, hẻm",3,#e45\r\n🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TI...,🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TIỆN XÂY M...,"Đường Xô Viết Nghệ Tĩnh, Phường 21, Quận Bình ...","[159.22, 159.22, 157.45, 157.45, 135.08, 122.2...",Bình Thạnh,150.91,141.233077,-8.31,13.531587
4,đã có sổ,18.0,2.85,"Nhà ngõ, hẻm",2,Nhà hẻm cách đường mặt tiền 20m\r\nNhà ở khu v...,Nhà hẻm cách đường mặt tiền 20m,"Đường Chu Văn An, Phường 12, Quận Bình Thạnh, ...","[132.67, 132.67, 169.64, 169.64, 133.33, 133.3...",Bình Thạnh,171.74,149.880000,39.07,15.542345


In [10]:
df = df[~df[["gia_kv_hien_tai", "gia_kv_mean", "gia_kv_trend", "gia_kv_volatility"]].isnull().any(axis=1)]

In [11]:
df.shape

(6979, 14)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6979 entries, 0 to 7960
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   giay_to_phap_ly    6979 non-null   object 
 1   dien_tich          6979 non-null   float64
 2   gia_ban            6979 non-null   float64
 3   loai_hinh          6979 non-null   object 
 4   so_phong_ngu       6979 non-null   int64  
 5   mo_ta              6979 non-null   object 
 6   tieu_de            6979 non-null   object 
 7   dia_chi            6979 non-null   object 
 8   bieu_do_gia        6979 non-null   object 
 9   quan               6979 non-null   object 
 10  gia_kv_hien_tai    6979 non-null   float64
 11  gia_kv_mean        6979 non-null   float64
 12  gia_kv_trend       6979 non-null   float64
 13  gia_kv_volatility  6979 non-null   float64
dtypes: float64(6), int64(1), object(7)
memory usage: 817.9+ KB


### Giải thích: Tại sao xóa 1294 rows?

Ở cell trên, dòng `df = df[~df[["gia_kv_hien_tai", "gia_kv_mean", "gia_kv_trend", "gia_kv_volatility"]].isnull().any(axis=1)]` đã loại bỏ **1294 rows** vì:

**Nguyên nhân:** 4 cột trên được parse từ cột `bieu_do_gia` (biểu đồ giá khu vực). Hàm `parse_bieu_do_gia()` trả về `[]` (rỗng) khi:
- `bieu_do_gia` bị **null/NaN** → không có dữ liệu giá khu vực
- Giá trị **không phải list hợp lệ** (format sai, không parse được)
- List có **ít hơn 2 phần tử** → không đủ dữ liệu để tính trend/volatility

→ Khi `bdg_parsed` rỗng, 4 features (`gia_kv_hien_tai`, `gia_kv_mean`, `gia_kv_trend`, `gia_kv_volatility`) đều thành **NaN**.

**Tại sao phải xóa thay vì impute?**
- 4 features này mang thông tin **giá thị trường khu vực** — rất quan trọng cho model dự đoán giá
- Không thể impute hợp lý vì mỗi khu vực có biểu đồ giá riêng biệt
- Giữ lại rows thiếu sẽ gây **noise** cho model (giá trị giả không phản ánh thực tế)
- 1294 rows (~vài % dataset) là mức chấp nhận được, không ảnh hưởng nhiều đến tổng data

## 3. Trích xuất features từ Text (NLP đơn giản)

Sử dụng Regex để trích xuất các đặc điểm quan trọng từ tiêu đề + mô tả:
- **Vị trí & Lưu thông**: mặt tiền, hẻm xe hơi, lô góc
- **Tiềm năng kinh tế**: kinh doanh, cho thuê/dòng tiền
- **Đặc tính nhà**: nở hậu, thang máy, nhà mới/nát, sân thượng, gara,...
- **Tín hiệu bất thường**: ngộp bank, bán gấp (dùng cho Anomaly Detection)
- **Tiện nghi score**: đếm số từ khóa tiện ích xuất hiện


In [13]:
# Gộp tiêu đề và mô tả thành một chuỗi duy nhất, chuyển về chữ thường để dễ Regex
df['full_text'] = (df['tieu_de'].fillna('') + ' ' + df['mo_ta'].fillna('')).str.lower()

In [14]:
# Vị trí & Lưu thông
df['is_mat_tien'] = df['full_text'].str.contains(r'mặt tiền|mt |mặt đường|mt', regex=True).astype(int)
df['is_hxh'] = df['full_text'].str.contains(r'hẻm xe hơi|hxh|hẻm ô tô|hẻm oto|hẻm 6m|hẻm 8m|hẻm rộng', regex=True).astype(int)
df['is_lo_goc'] = df['full_text'].str.contains(r'lô góc|góc 2 mặt|2 mặt tiền|3 mặt tiền|nhà góc|2mt|3mt|hai mặt|ba mặt', regex=True).astype(int)

In [15]:
# Tiềm năng kinh tế
df['is_kinh_doanh'] = df['full_text'].str.contains(r'kinh doanh|buôn bán|mặt bằng', regex=True).astype(int)
df['is_dong_tien'] = df['full_text'].str.contains(r'cho thuê|dòng tiền|thu nhập|chdv|căn hộ dịch vụ', regex=True).astype(int)

In [16]:
# Đặc tính & Phong thủy
df['is_no_hau'] = df['full_text'].str.contains(r'nở hậu', regex=True).astype(int)
df['has_thang_may'] = df['full_text'].str.contains(r'thang máy', regex=True).astype(int)
df['is_nha_moi'] = df['full_text'].str.contains(r'nhà mới|mới đẹp|full nội thất|xách vali', regex=True).astype(int)
df['is_nha_nat'] = df['full_text'].str.contains(r'nhà nát|cũ nát|tiện xây mới|bán đất|đập đi', regex=True).astype(int)
df['has_quy_hoach'] = df['full_text'].str.contains(r'quy hoạch|lộ giới|dính quy', regex=True).astype(int)
df['is_chinh_chu'] = df['full_text'].str.contains(r'chính chủ|1 đời chủ|đời chủ', regex=True).astype(int)
df['has_san_thuong'] = df['full_text'].str.contains(r'sân thượng', regex=True).astype(int)
df['has_gara'] = df['full_text'].str.contains(r'gara|ga ra|garage|ô tô đỗ|xe hơi ngủ|ô tô vào nhà|xe hơi vào', regex=True).astype(int)
df['o_ngay'] = df['full_text'].str.contains(r'ở ngay|ở liền|dọn vào ở|vào ở ngay', regex=True).astype(int)


In [17]:
# Tín hiệu Gấp/Ngộp (Cho Anomaly Detection)
df['is_ngop_bank'] = df['full_text'].str.contains(r'ngộp|ngân hàng|giải chấp|vỡ nợ|phá sản|vay bank', regex=True).astype(int)
df['is_ban_gap'] = df['full_text'].str.contains(r'bán gấp|hạ chào|giảm giá|cắt lỗ|giảm sâu|cần tiền', regex=True).astype(int)

In [18]:
# Tiện nghi score
tien_ich_kw = ["gara","garage","hồ bơi","sân thượng","thang máy","sân vườn",
               "ban công","giếng trời","ô tô","xe hơi","kinh doanh",
               "trường học","bệnh viện","chợ","công viên","siêu thị"]
df["tien_nghi_score"] = df['full_text'].apply(
    lambda x: sum(1 for kw in tien_ich_kw if kw in str(x)))

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6979 entries, 0 to 7960
Data columns (total 32 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   giay_to_phap_ly    6979 non-null   object 
 1   dien_tich          6979 non-null   float64
 2   gia_ban            6979 non-null   float64
 3   loai_hinh          6979 non-null   object 
 4   so_phong_ngu       6979 non-null   int64  
 5   mo_ta              6979 non-null   object 
 6   tieu_de            6979 non-null   object 
 7   dia_chi            6979 non-null   object 
 8   bieu_do_gia        6979 non-null   object 
 9   quan               6979 non-null   object 
 10  gia_kv_hien_tai    6979 non-null   float64
 11  gia_kv_mean        6979 non-null   float64
 12  gia_kv_trend       6979 non-null   float64
 13  gia_kv_volatility  6979 non-null   float64
 14  full_text          6979 non-null   object 
 15  is_mat_tien        6979 non-null   int64  
 16  is_hxh             6979 non-n

### Xóa các cột text thô (đã trích xuất xong features)

| Cột bị xóa | Lý do | Đã chuyển thành |
|---|---|---|
| `tieu_de` | Text tự do, model không dùng trực tiếp | Gộp vào `full_text` → trích ~15 features nhị phân (`is_mat_tien`, `is_hxh`, `is_ban_gap`,...) + `tien_nghi_score` |
| `mo_ta` | Text dài, tương tự tiêu đề | Gộp vào `full_text` → trích features tương tự |
| `dia_chi` | Địa chỉ không đồng nhất | Trích thành cột `quan` (tên quận chuẩn hóa) qua `extract_quan()` |
| `bieu_do_gia` | List dạng string `"[50, 55, 60]"` | Parse thành 4 features: `gia_kv_hien_tai`, `gia_kv_mean`, `gia_kv_trend`, `gia_kv_volatility` |

> 

In [20]:
drop_cols = [
    "tieu_de",
    "mo_ta",
    "dia_chi",
    "bieu_do_gia",
]

df = df.drop(columns=drop_cols)


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6979 entries, 0 to 7960
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   giay_to_phap_ly    6979 non-null   object 
 1   dien_tich          6979 non-null   float64
 2   gia_ban            6979 non-null   float64
 3   loai_hinh          6979 non-null   object 
 4   so_phong_ngu       6979 non-null   int64  
 5   quan               6979 non-null   object 
 6   gia_kv_hien_tai    6979 non-null   float64
 7   gia_kv_mean        6979 non-null   float64
 8   gia_kv_trend       6979 non-null   float64
 9   gia_kv_volatility  6979 non-null   float64
 10  full_text          6979 non-null   object 
 11  is_mat_tien        6979 non-null   int64  
 12  is_hxh             6979 non-null   int64  
 13  is_lo_goc          6979 non-null   int64  
 14  is_kinh_doanh      6979 non-null   int64  
 15  is_dong_tien       6979 non-null   int64  
 16  is_no_hau          6979 non-n

## 4. Log-transform giá bán

Giá nhà thường có phân phối lệch phải (right-skewed). Biến đổi log giúp:
- Phân phối gần chuẩn hơn → tốt cho Linear Regression
- Giảm ảnh hưởng của outliers
- Sai số trở thành tương đối (% thay vì tuyệt đối)


In [22]:
df["log_gia_ban"] = np.log(df["gia_ban"])

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6979 entries, 0 to 7960
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   giay_to_phap_ly    6979 non-null   object 
 1   dien_tich          6979 non-null   float64
 2   gia_ban            6979 non-null   float64
 3   loai_hinh          6979 non-null   object 
 4   so_phong_ngu       6979 non-null   int64  
 5   quan               6979 non-null   object 
 6   gia_kv_hien_tai    6979 non-null   float64
 7   gia_kv_mean        6979 non-null   float64
 8   gia_kv_trend       6979 non-null   float64
 9   gia_kv_volatility  6979 non-null   float64
 10  full_text          6979 non-null   object 
 11  is_mat_tien        6979 non-null   int64  
 12  is_hxh             6979 non-null   int64  
 13  is_lo_goc          6979 non-null   int64  
 14  is_kinh_doanh      6979 non-null   int64  
 15  is_dong_tien       6979 non-null   int64  
 16  is_no_hau          6979 non-n

### Nhận xét
Dataset cuối cùng có **6,979 dòng** và **29 cột** (28 features + 1 target `log_gia_ban`).
Sẵn sàng cho bước modeling.


In [24]:
df.shape

(6979, 29)

## 5. Lưu dataset cho Modeling


In [25]:
df.to_csv("../data/processed/data_modeling.csv", index=False, encoding="utf-8-sig")

In [26]:
df.to_parquet("../data/processed/data_modeling.parquet", index=False)


In [27]:
import os

print(os.listdir("../data/processed/"))

['dataset_cleaned.csv', 'dataset_cleaned.parquet', 'dataset_merged.csv', 'data_modeling.csv', 'data_modeling.parquet', 'pyspark_predictions.parquet']


In [28]:
test_df = pd.read_csv('../data/processed/data_modeling.csv')

In [29]:
test_df.shape

(6979, 29)

In [30]:
test_df = pd.read_parquet('../data/processed/data_modeling.parquet')

In [31]:
test_df.shape

(6979, 29)

## Kết luận

### Tổng kết Feature Engineering

| Nhóm Feature | Số lượng | Phương pháp | Ví dụ |
|---|---|---|---|
| Quận (categorical) | 1 | Regex trích xuất từ `dia_chi` | Gò Vấp, Bình Thạnh, Phú Nhuận |
| Giá khu vực (numeric) | 4 | Parse từ `bieu_do_gia` | `gia_kv_mean`, `gia_kv_trend`,... |
| Vị trí & Lưu thông (binary) | 3 | Regex từ text | `is_mat_tien`, `is_hxh`, `is_lo_goc` |
| Tiềm năng kinh tế (binary) | 2 | Regex từ text | `is_kinh_doanh`, `is_dong_tien` |
| Đặc tính nhà (binary) | 9 | Regex từ text | `is_nha_moi`, `has_thang_may`,... |
| Tín hiệu bất thường (binary) | 2 | Regex từ text | `is_ngop_bank`, `is_ban_gap` |
| Tiện nghi score (numeric) | 1 | Đếm từ khóa | `tien_nghi_score` |
| Log giá bán (numeric) | 1 | `np.log(gia_ban)` | `log_gia_ban` |

### Dataset cuối cùng
- **Kích thước**: 6,979 dòng x 29 cột
- **Giảm**: 982 dòng so với dataset cleaned (do thiếu `bieu_do_gia` hợp lệ)
- **Features numeric**: dien_tich, so_phong_ngu, gia_kv_*, tien_nghi_score, log_gia_ban
- **Features binary**: 16 features (is_mat_tien, is_hxh,...)
- **Features categorical**: giay_to_phap_ly, loai_hinh, quan
- **Target**: `log_gia_ban` (log của giá bán, đơn vị tỷ VNĐ)

### Điểm mạnh của bộ features
1. **Đa dạng nguồn thông tin**: kết hợp dữ liệu số, categorical, và text
2. **Domain-specific**: các features được thiết kế dựa trên kiến thức thị trường BĐS VN
3. **Phục vụ cả 2 bài toán**: features cho Regression (giá khu vực, tiện nghi) và Anomaly Detection (ngộp bank, bán gấp)

### File output
- `data/processed/data_modeling.csv`
- `data/processed/data_modeling.parquet`
